# Step 1: Data Loading and normalization

In [ ]:
# Install sklearn if not already installed
%pip install scikit-learn

   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/11.2 MB 6.7 MB/s eta 0:00:02
   --------- ------------------------------ 2.6/11.2 MB 6.9 MB/s eta 0:00:02
   --------------- ------------------------ 4.2/11.2 MB 7.0 MB/s eta 0:00:01
   ------------------- -------------------- 5.5/11.2 MB 6.8 MB/s eta 0:00:01
   ------------------------- -------------- 7.1/11.2 MB 6.9 MB/s eta 0:00:01
   ------------------------------ --------- 8.4/11.2 MB 6.9 MB/s eta 0:00:01
   ---------------------------------- ----- 9.7/11.2 MB 6.9 MB/s eta 0:00:01
   ---------------------------------------  11.0/11.2 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 11.2/11.2 MB 6.6 MB/s  0:00:01
   ---------------------------------------- 0.0/15.9 MB ? eta -:--:--
   --- ------------------------------------ 1.3/15.9 MB 6.7 MB/s eta 0:00:03
   ------- -------------------------------- 2.9/15.9 MB 6.7 MB/s eta 0:00:02
   ---------- 

In [ ]:
# Install pywt if not already installed
%pip install PyWavelets

   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   --------- ------------------------------ 1.0/4.3 MB 6.3 MB/s eta 0:00:01
   ---------------------- ----------------- 2.4/4.3 MB 5.8 MB/s eta 0:00:01
   -------------------------------- ------- 3.4/4.3 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 4.3/4.3 MB 5.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Install tensorflow if not already installed
%pip install tensorflow

   ---------------------------------------- 0.0/331.7 MB ? eta -:--:--
   ---------------------------------------- 1.3/331.7 MB 6.7 MB/s eta 0:00:50
   ---------------------------------------- 2.6/331.7 MB 6.9 MB/s eta 0:00:48
    --------------------------------------- 4.2/331.7 MB 7.0 MB/s eta 0:00:47
    --------------------------------------- 5.8/331.7 MB 6.9 MB/s eta 0:00:48
    --------------------------------------- 6.8/331.7 MB 6.9 MB/s eta 0:00:48
    --------------------------------------- 8.1/331.7 MB 6.8 MB/s eta 0:00:48
   - -------------------------------------- 9.7/331.7 MB 6.7 MB/s eta 0:00:48
   - -------------------------------------- 11.0/331.7 MB 6.7 MB/s eta 0:00:48
   - -------------------------------------- 12.6/331.7 MB 6.8 MB/s eta 0:00:47
   - -------------------------------------- 14.2/331.7 MB 6.8 MB/s eta 0:00:47
   - -------------------------------------- 15.5/331.7 MB 6.9 MB/s eta 0:00:47
   -- ------------------------------------- 17.0/331.7 MB 6.9 MB/s 

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

# Paths to your data folders
# Make sure you are running this notebook in the same directory as the data folders
truth_path = '../LieWaves/Truth_Sessions/1_BandPass_Filtered/'
lie_path = '../LieWaves/Lie_Sessions/1_BandPass_Filtered/'

# Function to load all CSVs and concatenate them into a single DataFrame
def load_and_concatenate_data(path):
    data_frames = []
    for filename in os.listdir(path):
        if filename.endswith('.csv'):
            df = pd.read_csv(os.path.join(path, filename))
            data_frames.append(df)
    concatenated_df = pd.concat(data_frames, ignore_index=True)
    return concatenated_df

truth_data = load_and_concatenate_data(truth_path)
lie_data = load_and_concatenate_data(lie_path)

# Function to normalize data
scaler = StandardScaler()
def normalize_data(df):
    return scaler.fit_transform(df)

truth_data = normalize_data(truth_data)
lie_data = normalize_data(lie_data)



# Step 2: Segmenting the Data

In [ ]:
# Function to segment data
def segment_data(df, window_size=128, overlap=64):
    segments = []
    for start in range(0, len(df) - window_size, overlap):
        segment = df[start:start + window_size, :]
        segments.append(segment)
    return np.array(segments)

# Segment the data
window_size = 128  # 1 second of data
overlap = 64  # 50% overlap

truth_segments = segment_data(truth_data, window_size, overlap)
lie_segments = segment_data(lie_data, window_size, overlap)

# Step 3: Feature Extraction with DWT

In [ ]:
import pywt

# Function to extract DWT features
def extract_dwt_features(segments, wavelet='db4', level=4):
    features = []
    for segment in segments:
        segment_features = []
        for channel in range(segment.shape[1]):
            coeffs = pywt.wavedec(segment[:, channel], wavelet, level=level)
            coeffs_flat = np.hstack(coeffs)
            segment_features.append(coeffs_flat)
        features.append(np.hstack(segment_features))
    return np.array(features)

# Extract DWT features
truth_features = extract_dwt_features(truth_segments)
lie_features = extract_dwt_features(lie_segments)

# Step 4: Preparing Data for CNN

In [ ]:
# Create labels: 1 for truth, 0 for lie
truth_labels = np.ones(truth_features.shape[0])
lie_labels = np.zeros(lie_features.shape[0])

# Combine features and labels
X = np.vstack((truth_features, lie_features))
y = np.hstack((truth_labels, lie_labels))

# Split data into training and testing sets
from sklearn.model_selection import train_test_split

# Ensure both classes are in the test set by using stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Verify the split
print("Training set class distribution:", np.bincount(y_train.astype(int)))
print("Test set class distribution:", np.bincount(y_test.astype(int)))

Training set class distribution: [3238 3238]
Test set class distribution: [810 810]


# Step 5: Define and Train the CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models,callbacks

# Define the CNN model
model = models.Sequential()

# Convolution Layers
# Stage 1
model.add(layers.Conv1D(256, 3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(tf.keras.layers.BatchNormalization())
model.add(layers.MaxPooling1D(2))
model.add(layers.Dropout(0.25))

# Stage 2
model.add(layers.Conv1D(128, 3, activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(layers.MaxPooling1D(2))
model.add(layers.Dropout(0.25))

# Stage 3
model.add(layers.Conv1D(64, 3, activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(layers.MaxPooling1D(2))
model.add(layers.Dropout(0.25))

# Flatten
model.add(layers.Flatten())

# Fully Connected Layers
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(2, activation='softmax'))

# Output
model.add(layers.Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Reshape data for the CNN
X_train_cnn = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Define callbacks for early stopping and model checkpointing
callbacks = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint('best_model.keras', save_best_only=True)
]

# Train the model
history = model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, validation_data=(X_test_cnn, y_test), callbacks=callbacks)

# Load the best model
model.load_weights('best_model.keras')

c:\Users\aroon\Desktop\Masterarbeit\.venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 36s 157ms/step - accuracy: 0.6325 - loss: 0.5938 - val_accuracy: 0.4994 - val_loss: 0.7001
Epoch 2/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 30s 148ms/step - accuracy: 0.8924 - loss: 0.5159 - val_accuracy: 0.6969 - val_loss: 0.6004
Epoch 3/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 31s 151ms/step - accuracy: 0.8708 - loss: 0.4863 - val_accuracy: 0.8519 - val_loss: 0.4806
Epoch 4/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - accuracy: 0.8851 - loss: 0.4512 - val_accuracy: 0.8735 - val_loss: 0.4377
Epoch 5/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - accuracy: 0.8808 - loss: 0.4315 - val_accuracy: 0.8074 - val_loss: 0.4968
Epoch 6/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - accuracy: 0.8891 - loss: 0.4040 - val_accuracy: 0.7290 - val_loss: 0.5836
Epoch 7/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - accuracy: 0.8844 - loss: 0.3949 - val_accuracy: 0.8988 - val_loss: 0.3659
Epoch 8/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 30s 148ms/step - accuracy: 0.8947 - loss: 0

# Step 6: Evaluate the Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Make predictions
y_pred = model.predict(X_test_cnn)
y_pred_classes = (y_pred > 0.5).astype("int32")

# Evaluate the model
print(classification_report(y_test, y_pred_classes, zero_division=1))
print("F1 Score:", f1_score(y_test, y_pred_classes, zero_division=1))


51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step
              precision    recall  f1-score   support

         0.0       0.99      0.84      0.91       810
         1.0       0.86      1.00      0.92       810

    accuracy                           0.92      1620
   macro avg       0.93      0.92      0.92      1620
weighted avg       0.93      0.92      0.92      1620

F1 Score: 0.9237822349570202
